In [3]:
#!/usr/bin/env python3
# ant_late_cycle_separation_tests_layerwise_strict.py
# Layer-by-layer late-cycle (cycles 15–50) self-vs-task separation tests for:
#   (1) Continual Learning (CL): walk/jump/spin continual
#   (2) Walk-only baseline
#
# For each layer L:
#   Δ_r,L = mean_{b, t in [15,50)} [ self(r,b,L,t) - task(r,b,L,t) ]   (one number per run)
#
# Stringent hypotheses (examples):
#   H1: mean(Δ_r,L) > 0.10   (≥ 10 percentage points separation)
#   H1: mean(Δ_r,L) > 0.15   (≥ 15 percentage points separation)
#
# Tests:
#   - Within-condition: {one-sample t-test} one-sided, H1: mean(Δ_r,L) > threshold
#   - CL vs Walk: paired if run_name overlaps else {Welch’s t-test} one-sided,
#       H1: mean(Δ_CL - Δ_WALK) > threshold

import os
import glob
import re
import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

ROBUSTNESS_ROOT_CL   = "/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/WalkSpinJump_relu"
ROBUSTNESS_ROOT_WALK = "/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/Walk_robustness"

LATE_START_CYCLE = 15
LATE_END_CYCLE   = 50  # exclusive

# More stringent separation thresholds (in persistence-score units, 0..1)
# 0.10 means "10 percentage points"
SEPARATION_THRESHOLDS = [0.10, 0.15]

# If True: CL vs Walk uses paired test when run_name overlaps.
PREFER_PAIRED_IF_POSSIBLE = True

# If "auto": uses behaviors present in each dataset.
BEHAVIORS = "auto"

# ============================================================
# Helpers (cache discovery + load)
# ============================================================

def _find_phase_csvs(root: str):
    patterns = [
        os.path.join(root, "*", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "*", "organized", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "organized", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "phase_summary.csv"),
    ]
    out = []
    for pat in patterns:
        out.extend(glob.glob(pat))
    return sorted(set(out))

def _run_name_from_csv_path(csv_path: str, root_base: str) -> str:
    parts = os.path.normpath(csv_path).split(os.sep)
    try:
        idx = parts.index(root_base)
        return parts[idx + 1]
    except Exception:
        if "organized" in parts:
            return parts[parts.index("organized") - 1]
        if "models" in parts:
            return parts[parts.index("models") - 1]
        return os.path.basename(os.path.dirname(os.path.dirname(csv_path)))

def _safe_read_phase_csv(csv_path: str, min_bytes: int = 5):
    if not os.path.exists(csv_path):
        return None, "missing file"
    if os.path.getsize(csv_path) < min_bytes:
        return None, f"empty file (<{min_bytes} bytes)"
    try:
        df = pd.read_csv(csv_path)
    except EmptyDataError:
        return None, "EmptyDataError (no columns to parse)"
    except Exception as e:
        return None, f"read_csv failed: {type(e).__name__}: {e}"

    if df is None or df.shape[1] == 0:
        return None, "no columns after read"
    if len(df) == 0:
        return None, "valid header but 0 rows"

    required_cols = ["behavior", "layer_idx", "cluster_rank"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        return None, f"missing required cols: {missing}"

    return df, None

def _parse_cycle_num_from_cycle_id(cycle_id: str) -> int:
    m = re.search(r"(\d+)", str(cycle_id))
    return int(m.group(1)) if m else -1

def _ensure_cycle_num_column(df: pd.DataFrame) -> pd.DataFrame:
    if "cycle_num" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle_num"], errors="coerce").fillna(-1).astype(int)
        return out
    if "cycle_idx" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle_idx"], errors="coerce").fillna(-1).astype(int)
        return out
    if "cycle" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle"], errors="coerce").fillna(-1).astype(int)
        return out
    if "cycle_id" in df.columns:
        out = df.copy()
        out["cycle_num"] = out["cycle_id"].apply(_parse_cycle_num_from_cycle_id).astype(int)
        return out
    if "step_idx" in df.columns:
        out = df.copy()
        step = pd.to_numeric(out["step_idx"], errors="coerce").fillna(-1).astype(int)
        out["cycle_num"] = step
        return out
    raise RuntimeError(
        "Could not determine cycle index. Expected one of: cycle_id / cycle_num / cycle_idx / cycle / step_idx."
    )

# ============================================================
# Build dfm (module-level self/task) from phase_summary.csv
# ============================================================

def _load_dfm_from_robustness_root(root: str, behaviors="auto"):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"ROBUSTNESS_ROOT not found: {root}")

    phase_csvs = _find_phase_csvs(root)
    if not phase_csvs:
        raise FileNotFoundError(f"No phase_summary.csv found under: {root}")

    root_base = os.path.basename(os.path.normpath(root))

    dfs = []
    skipped = []
    for csv_path in phase_csvs:
        run_name = _run_name_from_csv_path(csv_path, root_base=root_base)
        df, reason = _safe_read_phase_csv(csv_path)
        if df is None:
            skipped.append((run_name, csv_path, reason))
            continue

        df = df.copy()
        df["run_name"] = run_name
        try:
            df = _ensure_cycle_num_column(df)
        except Exception as e:
            skipped.append((run_name, csv_path, f"cycle inference failed: {type(e).__name__}: {e}"))
            continue

        dfs.append(df)

    print(f"\n[LOAD] root={root}")
    print(f"[LOAD] Found {len(phase_csvs)} phase_summary.csv files")
    print(f"[LOAD] Loaded {len(dfs)} runs successfully")
    print(f"[LOAD] Skipped {len(skipped)} runs")

    if not dfs:
        raise RuntimeError(f"No valid runs loaded from: {root}")

    df_all = pd.concat(dfs, ignore_index=True)

    dfp = df_all.copy()
    dfp["behavior"] = dfp["behavior"].astype(str).str.strip().str.lower()

    _PREF_ORDER = ["walk", "jump", "spin"]
    present = [b for b in _PREF_ORDER if b in set(dfp["behavior"].unique().tolist())]

    if behaviors == "auto":
        if not present:
            present = sorted(set(dfp["behavior"].unique().tolist()))
            if not present:
                raise RuntimeError("No behaviors found in data.")
        BEH_USED = present
    else:
        BEH_USED = [str(b).strip().lower() for b in list(behaviors)]
        BEH_USED = [b for b in BEH_USED if b in set(dfp["behavior"].unique().tolist())]
        if not BEH_USED:
            raise RuntimeError(f"Requested behaviors={behaviors} but none are present in the data.")

    dfp = dfp[dfp["behavior"].isin(BEH_USED)].copy()

    for c in ["layer_idx", "cluster_rank"]:
        dfp[c] = pd.to_numeric(dfp[c], errors="coerce").fillna(-1).astype(int)

    if "mean_persistence_score" in dfp.columns:
        SCORE_COL = "mean_persistence_score"
    elif "mean_self_score" in dfp.columns:
        SCORE_COL = "mean_self_score"
    elif "mean_persistence" in dfp.columns:
        SCORE_COL = "mean_persistence"
    else:
        raise KeyError(
            "Missing score column. Expected one of: mean_persistence_score / mean_self_score / mean_persistence."
        )
    dfp[SCORE_COL] = pd.to_numeric(dfp[SCORE_COL], errors="coerce")

    if "cluster_size_neurons" not in dfp.columns:
        raise KeyError("Missing cluster_size_neurons in CSV; needed to define self/task modules.")
    dfp["cluster_size_neurons"] = pd.to_numeric(dfp["cluster_size_neurons"], errors="coerce").fillna(0).astype(float)

    # ---- Build dfm exactly like your plotting code ----
    df_self = dfp[dfp["cluster_rank"] == 1].copy()
    if len(df_self) == 0:
        raise RuntimeError("No cluster_rank==1 rows found; cannot define self module.")
    df_self["module"] = "self"
    df_self_mod = df_self.copy()

    df_other = dfp[dfp["cluster_rank"] >= 2].copy()
    df_other["w"] = df_other["cluster_size_neurons"] * df_other[SCORE_COL]

    group_keys = ["run_name", "behavior", "cycle_num", "layer_idx"]

    task_agg = df_other.groupby(group_keys, as_index=False).agg(
        other_size_sum=("cluster_size_neurons", "sum"),
        w_sum=("w", "sum"),
    )

    task_agg["module"] = "task"
    task_agg[SCORE_COL] = task_agg["w_sum"] / task_agg["other_size_sum"].replace(0, np.nan)
    task_agg["cluster_size_neurons"] = task_agg["other_size_sum"]

    if "n_alive_units" in df_self_mod.columns:
        alive_info = df_self_mod[group_keys + ["n_alive_units", "cluster_size_neurons"]].copy()
        alive_info = alive_info.rename(columns={
            "cluster_size_neurons": "self_size",
            "n_alive_units": "n_alive_units",
        })
        alive_info["n_alive_units"] = pd.to_numeric(alive_info["n_alive_units"], errors="coerce")
        alive_info["self_size"] = pd.to_numeric(alive_info["self_size"], errors="coerce")

        task_agg = task_agg.merge(alive_info, on=group_keys, how="left")
        task_agg["cluster_size_neurons"] = (task_agg["n_alive_units"] - task_agg["self_size"]).clip(lower=0)

    task_rows = task_agg[group_keys + ["module", "cluster_size_neurons", SCORE_COL]].copy()

    dfm = pd.concat(
        [
            df_self_mod[group_keys + ["module", "cluster_size_neurons", SCORE_COL]].copy(),
            task_rows.copy(),
        ],
        ignore_index=True
    )

    dfm["module"] = dfm["module"].astype(str).str.strip().str.lower()

    # Drop incomplete cycles (exactly like your code)
    N_BEH_USED = len(BEH_USED)
    grp = dfm.groupby(["run_name", "cycle_num", "layer_idx", "module"])["behavior"].nunique()
    valid_keys = grp[grp == N_BEH_USED].index
    valid_df = valid_keys.to_frame(index=False)

    dfm = dfm.merge(
        valid_df,
        on=["run_name", "cycle_num", "layer_idx", "module"],
        how="inner"
    ).copy()

    if len(dfm) == 0:
        raise RuntimeError("All cycles dropped as incomplete; adjust BEHAVIORS or check the CSV content.")

    layers = sorted(dfm["layer_idx"].unique().tolist())
    print(f"[LOAD] Behaviors used: {BEH_USED} (N={N_BEH_USED})")
    print(f"[LOAD] Runs: {dfm['run_name'].nunique()} | Rows(dfm): {len(dfm)} | Layers: {layers}")

    return dfm, SCORE_COL, BEH_USED, layers

# ============================================================
# Δ_r,L and tests
# ============================================================

def _compute_delta_per_run_layer(dfm: pd.DataFrame, score_col: str, layer_idx: int, late_start: int, late_end: int):
    df = dfm.copy()
    df = df[(df["cycle_num"] >= late_start) & (df["cycle_num"] < late_end)]
    df = df[df["module"].isin(["self", "task"])]
    df = df[df["layer_idx"] == int(layer_idx)]

    wide = df.pivot_table(
        index=["run_name", "behavior", "layer_idx", "cycle_num"],
        columns="module",
        values=score_col,
        aggfunc="mean",
    ).dropna(subset=["self", "task"])

    if len(wide) == 0:
        return pd.Series(dtype=float)

    wide["delta"] = wide["self"] - wide["task"]

    d_run = wide.groupby("run_name")["delta"].mean()
    d_run = d_run.replace([np.inf, -np.inf], np.nan).dropna()
    return d_run

def _one_sample_one_sided_test_gt_threshold(deltas: pd.Series, threshold: float):
    x = np.asarray(deltas.values, dtype=float)
    x = x[np.isfinite(x)]
    n = int(x.size)
    if n < 2:
        return {"n": n, "mean": float(np.nanmean(x)) if n else np.nan, "sd": np.nan, "t_stat": np.nan, "p_one_sided": np.nan}

    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    se = sd / np.sqrt(n)

    # H1: mean(x) > threshold  <=> mean(x - threshold) > 0
    t_stat = (mu - float(threshold)) / se
    p_one = float(stats.t.sf(t_stat, df=n - 1))

    return {"n": n, "mean": mu, "sd": sd, "t_stat": float(t_stat), "p_one_sided": p_one}

def _mean_ci95_two_sided(deltas: pd.Series):
    x = np.asarray(deltas.values, dtype=float)
    x = x[np.isfinite(x)]
    n = int(x.size)
    if n < 2:
        return {"n": n, "mean": float(np.nanmean(x)) if n else np.nan, "sd": np.nan, "ci95_low": np.nan, "ci95_high": np.nan}

    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    se = sd / np.sqrt(n)
    tcrit = float(stats.t.ppf(0.975, df=n - 1))
    return {"n": n, "mean": mu, "sd": sd, "ci95_low": mu - tcrit * se, "ci95_high": mu + tcrit * se}

def _compare_cl_vs_walk_thresholds(d_cl: pd.Series, d_walk: pd.Series, thresholds, prefer_paired: bool = True):
    d_cl = d_cl.replace([np.inf, -np.inf], np.nan).dropna()
    d_walk = d_walk.replace([np.inf, -np.inf], np.nan).dropna()

    out = {"mode": "", "n": 0}
    if prefer_paired:
        common = sorted(set(d_cl.index) & set(d_walk.index))
        if len(common) >= 2:
            x = d_cl.loc[common].to_numpy(dtype=float)
            y = d_walk.loc[common].to_numpy(dtype=float)
            diff = x - y
            out["mode"] = "paired"
            out["n"] = int(len(common))
            for thr in thresholds:
                # H1: mean(diff) > thr  <=> mean(diff - thr) > 0
                res = stats.ttest_1samp(diff - float(thr), popmean=0.0, alternative="greater")
                out[f"p_gt_{thr:.2f}"] = float(res.pvalue)
                out[f"t_gt_{thr:.2f}"] = float(res.statistic)
            out["mean_diff"] = float(np.mean(diff))
            out["sd_diff"] = float(np.std(diff, ddof=1))
            return out

    # Welch (unpaired): H1 mean(CL) - mean(Walk) > thr  <=> mean(CL - thr) > mean(Walk)
    x = d_cl.to_numpy(dtype=float)
    y = d_walk.to_numpy(dtype=float)
    out["mode"] = "welch"
    out["n_cl"] = int(x.size)
    out["n_walk"] = int(y.size)
    out["mean_cl"] = float(np.mean(x)) if x.size else np.nan
    out["mean_walk"] = float(np.mean(y)) if y.size else np.nan
    out["sd_cl"] = float(np.std(x, ddof=1)) if x.size >= 2 else np.nan
    out["sd_walk"] = float(np.std(y, ddof=1)) if y.size >= 2 else np.nan
    for thr in thresholds:
        if x.size < 2 or y.size < 2:
            out[f"p_gt_{thr:.2f}"] = np.nan
            out[f"t_gt_{thr:.2f}"] = np.nan
            continue
        res = stats.ttest_ind(x - float(thr), y, equal_var=False, alternative="greater")
        out[f"p_gt_{thr:.2f}"] = float(res.pvalue)
        out[f"t_gt_{thr:.2f}"] = float(res.statistic)
    return out

# ============================================================
# MAIN
# ============================================================

dfm_cl, score_cl, beh_cl, layers_cl = _load_dfm_from_robustness_root(ROBUSTNESS_ROOT_CL, behaviors=BEHAVIORS)
dfm_walk, score_walk, beh_walk, layers_walk = _load_dfm_from_robustness_root(ROBUSTNESS_ROOT_WALK, behaviors=BEHAVIORS)

layers_to_use = sorted(set(layers_cl) & set(layers_walk)) if (layers_cl and layers_walk) else sorted(set(layers_cl + layers_walk))
if not layers_to_use:
    raise RuntimeError("No layers found across datasets.")

print("\n[LATE WINDOW]")
print(f"  cycles in [{LATE_START_CYCLE}, {LATE_END_CYCLE}) (i.e., {LATE_START_CYCLE}..{LATE_END_CYCLE-1})")
print(f"[STRICT THRESHOLDS] {SEPARATION_THRESHOLDS}")
print(f"[LAYERS] {layers_to_use}")

rows = []

for L in layers_to_use:
    d_cl = _compute_delta_per_run_layer(dfm_cl, score_cl, layer_idx=L, late_start=LATE_START_CYCLE, late_end=LATE_END_CYCLE)
    d_w  = _compute_delta_per_run_layer(dfm_walk, score_walk, layer_idx=L, late_start=LATE_START_CYCLE, late_end=LATE_END_CYCLE)

    cl_ci = _mean_ci95_two_sided(d_cl)
    w_ci  = _mean_ci95_two_sided(d_w)

    row = {
        "layer_idx": int(L),

        "cl_n": cl_ci["n"],
        "cl_mean": cl_ci["mean"],
        "cl_sd": cl_ci["sd"],
        "cl_ci95_low": cl_ci["ci95_low"],
        "cl_ci95_high": cl_ci["ci95_high"],

        "walk_n": w_ci["n"],
        "walk_mean": w_ci["mean"],
        "walk_sd": w_ci["sd"],
        "walk_ci95_low": w_ci["ci95_low"],
        "walk_ci95_high": w_ci["ci95_high"],
    }

    # Strict within-condition tests: mean(Δ) > threshold
    for thr in SEPARATION_THRESHOLDS:
        res_cl_thr = _one_sample_one_sided_test_gt_threshold(d_cl, threshold=thr)
        res_w_thr  = _one_sample_one_sided_test_gt_threshold(d_w, threshold=thr)

        row[f"cl_t_gt_{thr:.2f}"] = res_cl_thr["t_stat"]
        row[f"cl_p_gt_{thr:.2f}"] = res_cl_thr["p_one_sided"]

        row[f"walk_t_gt_{thr:.2f}"] = res_w_thr["t_stat"]
        row[f"walk_p_gt_{thr:.2f}"] = res_w_thr["p_one_sided"]

    # Strict CL vs Walk tests: mean(Δ_CL - Δ_WALK) > threshold
    cmp = _compare_cl_vs_walk_thresholds(d_cl, d_w, thresholds=SEPARATION_THRESHOLDS, prefer_paired=PREFER_PAIRED_IF_POSSIBLE)
    row["cmp_mode"] = cmp.get("mode", "")
    if cmp.get("mode", "") == "paired":
        row["cmp_n"] = cmp.get("n", 0)
        row["cmp_mean_diff"] = cmp.get("mean_diff", np.nan)
        row["cmp_sd_diff"] = cmp.get("sd_diff", np.nan)
    else:
        row["cmp_n_cl"] = cmp.get("n_cl", 0)
        row["cmp_n_walk"] = cmp.get("n_walk", 0)
        row["cmp_mean_diff"] = (cmp.get("mean_cl", np.nan) - cmp.get("mean_walk", np.nan))

    for thr in SEPARATION_THRESHOLDS:
        row[f"cmp_t_gt_{thr:.2f}"] = cmp.get(f"t_gt_{thr:.2f}", np.nan)
        row[f"cmp_p_gt_{thr:.2f}"] = cmp.get(f"p_gt_{thr:.2f}", np.nan)

    rows.append(row)

df_res = pd.DataFrame(rows).sort_values("layer_idx").reset_index(drop=True)

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 260)
pd.set_option("display.precision", 6)

print("\n" + "=" * 140)
print("LAYER-BY-LAYER LATE-CYCLE SEPARATION (Δ_r,L = mean(self-task) over cycles 15..49)")
print("Strict tests are one-sided: H1 mean(Δ) > threshold, and H1 mean(Δ_CL - Δ_WALK) > threshold.")
print("=" * 140)
print(df_res)
print("=" * 140)

print("\n[DONE]")


[LOAD] root=/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/WalkSpinJump_relu
[LOAD] Found 10 phase_summary.csv files
[LOAD] Loaded 10 runs successfully
[LOAD] Skipped 0 runs
[LOAD] Behaviors used: ['walk', 'jump', 'spin'] (N=3)
[LOAD] Runs: 10 | Rows(dfm): 4804 | Layers: [0, 1]

[LOAD] root=/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/Walk_robustness
[LOAD] Found 8 phase_summary.csv files
[LOAD] Loaded 8 runs successfully
[LOAD] Skipped 0 runs
[LOAD] Behaviors used: ['walk'] (N=1)
[LOAD] Runs: 8 | Rows(dfm): 4520 | Layers: [0, 1]

[LATE WINDOW]
  cycles in [15, 50) (i.e., 15..49)
[STRICT THRESHOLDS] [0.1, 0.15]
[LAYERS] [0, 1]

LAYER-BY-LAYER LATE-CYCLE SEPARATION (Δ_r,L = mean(self-task) over cycles 15..49)
Strict tests are one-sided: H1 mean(Δ) > threshold, and H1 mean(Δ_CL - Δ_WALK) > threshold.
   layer_idx  cl_n   cl_mean     cl_sd  cl_ci95_low  cl_ci95_high  walk_n  walk_mean   walk_sd  walk_ci95_low  walk_ci95_high  cl_t_gt_0.10

In [4]:
#!/usr/bin/env python3
# cl_vs_walk_strict_improvement_tests.py
#
# Computes, per layer:
#   1) Run-level late-cycle separation: Δ_r,l = mean_{b, t in [15,50)} (self - task)
#   2) “Probability” (one-sided p-value) that CL improves separation by at least a threshold τ:
#        H1: E[Δ_CL - Δ_WALK] > τ
#      using paired t-test if run_names overlap, else Welch’s t-test.
#   3) The improvement you can claim with 99% confidence:
#        99% one-sided lower confidence bound (LCB99) on E[Δ_CL - Δ_WALK]
#      plus 99% two-sided CI for completeness.
#
# Layer naming:
#   layer_idx=0 -> Layer 1
#   layer_idx=1 -> Layer 2

import os
import glob
import re
import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

ROBUSTNESS_ROOT_CL   = "/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/WalkSpinJump_relu"
ROBUSTNESS_ROOT_WALK = "/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/Walk_robustness"

LATE_START_CYCLE = 15
LATE_END_CYCLE   = 50  # exclusive (15..49)

# Thresholds you asked for:
#   Layer 1: 10% (0.10)
#   Layer 2:  5% (0.05)
THRESH_BY_LAYER_ONEBASED = {
    1: 0.10,  # Layer 1
    2: 0.05,  # Layer 2
}

ALPHA_ONE_SIDED = 0.01  # 99% one-sided LCB
ALPHA_TWO_SIDED = 0.01  # 99% two-sided CI

PREFER_PAIRED_IF_POSSIBLE = True
BEHAVIORS = "auto"

# ============================================================
# Helpers (cache discovery + load)
# ============================================================

def _find_phase_csvs(root: str):
    patterns = [
        os.path.join(root, "*", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "*", "organized", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "organized", "models", "_module_explorer_cache", "phase_summary.csv"),
        os.path.join(root, "phase_summary.csv"),
    ]
    out = []
    for pat in patterns:
        out.extend(glob.glob(pat))
    return sorted(set(out))

def _run_name_from_csv_path(csv_path: str, root_base: str) -> str:
    parts = os.path.normpath(csv_path).split(os.sep)
    try:
        idx = parts.index(root_base)
        return parts[idx + 1]
    except Exception:
        if "organized" in parts:
            return parts[parts.index("organized") - 1]
        if "models" in parts:
            return parts[parts.index("models") - 1]
        return os.path.basename(os.path.dirname(os.path.dirname(csv_path)))

def _safe_read_phase_csv(csv_path: str, min_bytes: int = 5):
    if not os.path.exists(csv_path):
        return None, "missing file"
    if os.path.getsize(csv_path) < min_bytes:
        return None, f"empty file (<{min_bytes} bytes)"
    try:
        df = pd.read_csv(csv_path)
    except EmptyDataError:
        return None, "EmptyDataError"
    except Exception as e:
        return None, f"read_csv failed: {type(e).__name__}: {e}"

    if df is None or df.shape[1] == 0:
        return None, "no columns"
    if len(df) == 0:
        return None, "0 rows"

    required_cols = ["behavior", "layer_idx", "cluster_rank"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        return None, f"missing required cols: {missing}"

    return df, None

def _parse_cycle_num_from_cycle_id(cycle_id: str) -> int:
    m = re.search(r"(\d+)", str(cycle_id))
    return int(m.group(1)) if m else -1

def _ensure_cycle_num_column(df: pd.DataFrame) -> pd.DataFrame:
    if "cycle_num" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle_num"], errors="coerce").fillna(-1).astype(int)
        return out
    if "cycle_id" in df.columns:
        out = df.copy()
        out["cycle_num"] = out["cycle_id"].apply(_parse_cycle_num_from_cycle_id).astype(int)
        return out
    if "cycle_idx" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle_idx"], errors="coerce").fillna(-1).astype(int)
        return out
    if "cycle" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["cycle"], errors="coerce").fillna(-1).astype(int)
        return out
    if "step_idx" in df.columns:
        out = df.copy()
        out["cycle_num"] = pd.to_numeric(out["step_idx"], errors="coerce").fillna(-1).astype(int)
        return out
    raise RuntimeError("Could not infer cycle_num.")

# ============================================================
# Build dfm (module-level self/task) from phase_summary.csv
# ============================================================

def _load_dfm_from_robustness_root(root: str, behaviors="auto"):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"ROBUSTNESS_ROOT not found: {root}")

    phase_csvs = _find_phase_csvs(root)
    if not phase_csvs:
        raise FileNotFoundError(f"No phase_summary.csv found under: {root}")

    root_base = os.path.basename(os.path.normpath(root))

    dfs = []
    skipped = []
    for csv_path in phase_csvs:
        run_name = _run_name_from_csv_path(csv_path, root_base=root_base)
        df, reason = _safe_read_phase_csv(csv_path)
        if df is None:
            skipped.append((run_name, csv_path, reason))
            continue

        df = df.copy()
        df["run_name"] = run_name
        try:
            df = _ensure_cycle_num_column(df)
        except Exception as e:
            skipped.append((run_name, csv_path, f"cycle inference failed: {type(e).__name__}: {e}"))
            continue

        dfs.append(df)

    print(f"\n[LOAD] root={root}")
    print(f"[LOAD] Found {len(phase_csvs)} phase_summary.csv files")
    print(f"[LOAD] Loaded {len(dfs)} runs successfully")
    print(f"[LOAD] Skipped {len(skipped)} runs")

    if not dfs:
        raise RuntimeError(f"No valid runs loaded from: {root}")

    df_all = pd.concat(dfs, ignore_index=True)

    dfp = df_all.copy()
    dfp["behavior"] = dfp["behavior"].astype(str).str.strip().str.lower()

    _PREF_ORDER = ["walk", "jump", "spin"]
    present = [b for b in _PREF_ORDER if b in set(dfp["behavior"].unique().tolist())]

    if behaviors == "auto":
        if not present:
            present = sorted(set(dfp["behavior"].unique().tolist()))
            if not present:
                raise RuntimeError("No behaviors found in data.")
        BEH_USED = present
    else:
        BEH_USED = [str(b).strip().lower() for b in list(behaviors)]
        BEH_USED = [b for b in BEH_USED if b in set(dfp["behavior"].unique().tolist())]
        if not BEH_USED:
            raise RuntimeError(f"Requested behaviors={behaviors} but none are present in the data.")

    dfp = dfp[dfp["behavior"].isin(BEH_USED)].copy()

    for c in ["layer_idx", "cluster_rank"]:
        dfp[c] = pd.to_numeric(dfp[c], errors="coerce").fillna(-1).astype(int)

    if "mean_persistence_score" in dfp.columns:
        SCORE_COL = "mean_persistence_score"
    elif "mean_self_score" in dfp.columns:
        SCORE_COL = "mean_self_score"
    elif "mean_persistence" in dfp.columns:
        SCORE_COL = "mean_persistence"
    else:
        raise KeyError("Missing score column.")

    dfp[SCORE_COL] = pd.to_numeric(dfp[SCORE_COL], errors="coerce")

    if "cluster_size_neurons" not in dfp.columns:
        raise KeyError("Missing cluster_size_neurons in CSV.")
    dfp["cluster_size_neurons"] = pd.to_numeric(dfp["cluster_size_neurons"], errors="coerce").fillna(0).astype(float)

    # Build self/task rows
    df_self = dfp[dfp["cluster_rank"] == 1].copy()
    if len(df_self) == 0:
        raise RuntimeError("No cluster_rank==1 rows found; cannot define self module.")
    df_self["module"] = "self"

    df_other = dfp[dfp["cluster_rank"] >= 2].copy()
    df_other["w"] = df_other["cluster_size_neurons"] * df_other[SCORE_COL]

    group_keys = ["run_name", "behavior", "cycle_num", "layer_idx"]

    task_agg = df_other.groupby(group_keys, as_index=False).agg(
        other_size_sum=("cluster_size_neurons", "sum"),
        w_sum=("w", "sum"),
    )
    task_agg["module"] = "task"
    task_agg[SCORE_COL] = task_agg["w_sum"] / task_agg["other_size_sum"].replace(0, np.nan)

    dfm = pd.concat(
        [
            df_self[group_keys + ["module", SCORE_COL]].copy(),
            task_agg[group_keys + ["module", SCORE_COL]].copy(),
        ],
        ignore_index=True
    )
    dfm["module"] = dfm["module"].astype(str).str.strip().str.lower()

    # Drop incomplete cycles (matches your robustness logic)
    N_BEH_USED = len(BEH_USED)
    grp = dfm.groupby(["run_name", "cycle_num", "layer_idx", "module"])["behavior"].nunique()
    valid_keys = grp[grp == N_BEH_USED].index
    dfm = dfm.merge(
        valid_keys.to_frame(index=False),
        on=["run_name", "cycle_num", "layer_idx", "module"],
        how="inner"
    ).copy()

    layers = sorted(dfm["layer_idx"].unique().tolist())
    print(f"[LOAD] Behaviors used: {BEH_USED} (N={N_BEH_USED})")
    print(f"[LOAD] Runs: {dfm['run_name'].nunique()} | Rows(dfm): {len(dfm)} | Layers: {layers}")

    return dfm, SCORE_COL, BEH_USED, layers

# ============================================================
# Δ_r,l extraction (run-level late-cycle mean separation)
# ============================================================

def _delta_per_run_per_layer(dfm: pd.DataFrame, score_col: str, layer_idx: int, late_start: int, late_end: int):
    df = dfm.copy()
    df = df[(df["cycle_num"] >= late_start) & (df["cycle_num"] < late_end)]
    df = df[(df["layer_idx"] == int(layer_idx)) & (df["module"].isin(["self", "task"]))]

    wide = df.pivot_table(
        index=["run_name", "behavior", "cycle_num"],
        columns="module",
        values=score_col,
        aggfunc="mean",
    ).dropna(subset=["self", "task"])

    if len(wide) == 0:
        return pd.Series(dtype=float)

    wide["delta"] = wide["self"] - wide["task"]

    # average across behaviors & cycles -> one Δ per run (for this layer)
    d_run = wide.groupby("run_name")["delta"].mean()
    return d_run.replace([np.inf, -np.inf], np.nan).dropna()

# ============================================================
# Stats: p-value for improvement > τ, and 99% bounds
# ============================================================

def _welch_df(sx2, nx, sy2, ny):
    num = (sx2 / nx + sy2 / ny) ** 2
    den = (sx2 * sx2) / (nx * nx * (nx - 1)) + (sy2 * sy2) / (ny * ny * (ny - 1))
    return float(num / den) if den > 0 else np.nan

def _compare_improvement_gt_tau(d_cl: pd.Series, d_w: pd.Series, tau: float, prefer_paired: bool = True):
    d_cl = d_cl.replace([np.inf, -np.inf], np.nan).dropna()
    d_w  = d_w.replace([np.inf, -np.inf], np.nan).dropna()

    # Paired if possible
    if prefer_paired:
        common = sorted(set(d_cl.index) & set(d_w.index))
        if len(common) >= 2:
            x = d_cl.loc[common].to_numpy(dtype=float)
            y = d_w.loc[common].to_numpy(dtype=float)
            diff = x - y
            # H1: mean(diff) > tau  <=> mean(diff - tau) > 0
            res = stats.ttest_1samp(diff - float(tau), popmean=0.0, alternative="greater")
            mean_diff = float(np.mean(diff))
            sd_diff = float(np.std(diff, ddof=1))
            n = int(diff.size)
            se = sd_diff / np.sqrt(n)
            df = n - 1

            # 99% one-sided LCB on mean(diff)
            tcrit_onesided = float(stats.t.ppf(1 - ALPHA_ONE_SIDED, df=df))
            lcb99 = mean_diff - tcrit_onesided * se

            # 99% two-sided CI on mean(diff)
            tcrit_twosided = float(stats.t.ppf(1 - ALPHA_TWO_SIDED / 2.0, df=df))
            ci_lo = mean_diff - tcrit_twosided * se
            ci_hi = mean_diff + tcrit_twosided * se

            return {
                "mode": "paired",
                "n_cl": n,
                "n_walk": n,
                "mean_cl": float(np.mean(x)),
                "mean_walk": float(np.mean(y)),
                "mean_diff": mean_diff,
                "p_one_sided_impr_gt_tau": float(res.pvalue),
                "t_stat_impr_gt_tau": float(res.statistic),
                "tau": float(tau),
                "lcb99_mean_diff": float(lcb99),
                "ci99_low": float(ci_lo),
                "ci99_high": float(ci_hi),
                "df_used": float(df),
            }

    # Welch (unpaired)
    x = d_cl.to_numpy(dtype=float)
    y = d_w.to_numpy(dtype=float)

    nx = int(x.size)
    ny = int(y.size)

    if nx < 2 or ny < 2:
        return {
            "mode": "welch",
            "n_cl": nx,
            "n_walk": ny,
            "mean_cl": float(np.nanmean(x)) if nx else np.nan,
            "mean_walk": float(np.nanmean(y)) if ny else np.nan,
            "mean_diff": np.nan,
            "p_one_sided_impr_gt_tau": np.nan,
            "t_stat_impr_gt_tau": np.nan,
            "tau": float(tau),
            "lcb99_mean_diff": np.nan,
            "ci99_low": np.nan,
            "ci99_high": np.nan,
            "df_used": np.nan,
        }

    # p-value for H1: mean(x) - mean(y) > tau  <=> mean(x - tau) > mean(y)
    res = stats.ttest_ind(x - float(tau), y, equal_var=False, alternative="greater")

    mean_x = float(np.mean(x))
    mean_y = float(np.mean(y))
    mean_diff = mean_x - mean_y

    sx2 = float(np.var(x, ddof=1))
    sy2 = float(np.var(y, ddof=1))
    se = float(np.sqrt(sx2 / nx + sy2 / ny))
    dfw = _welch_df(sx2, nx, sy2, ny)

    # 99% one-sided LCB on mean_diff
    tcrit_onesided = float(stats.t.ppf(1 - ALPHA_ONE_SIDED, df=dfw))
    lcb99 = mean_diff - tcrit_onesided * se

    # 99% two-sided CI on mean_diff
    tcrit_twosided = float(stats.t.ppf(1 - ALPHA_TWO_SIDED / 2.0, df=dfw))
    ci_lo = mean_diff - tcrit_twosided * se
    ci_hi = mean_diff + tcrit_twosided * se

    return {
        "mode": "welch",
        "n_cl": nx,
        "n_walk": ny,
        "mean_cl": mean_x,
        "mean_walk": mean_y,
        "mean_diff": mean_diff,
        "p_one_sided_impr_gt_tau": float(res.pvalue),
        "t_stat_impr_gt_tau": float(res.statistic),
        "tau": float(tau),
        "lcb99_mean_diff": float(lcb99),
        "ci99_low": float(ci_lo),
        "ci99_high": float(ci_hi),
        "df_used": float(dfw),
    }

# ============================================================
# MAIN
# ============================================================

dfm_cl, score_cl, beh_cl, layers_cl = _load_dfm_from_robustness_root(ROBUSTNESS_ROOT_CL, behaviors=BEHAVIORS)
dfm_w,  score_w,  beh_w,  layers_w  = _load_dfm_from_robustness_root(ROBUSTNESS_ROOT_WALK, behaviors=BEHAVIORS)

layers_to_use = sorted(set(layers_cl) & set(layers_w)) if (layers_cl and layers_w) else sorted(set(layers_cl + layers_w))
if not layers_to_use:
    raise RuntimeError("No layers found across datasets.")

print("\n[LATE WINDOW]")
print(f"  cycles in [{LATE_START_CYCLE}, {LATE_END_CYCLE}) (i.e., {LATE_START_CYCLE}..{LATE_END_CYCLE-1})")
print("[THRESHOLDS]")
for L1, tau in THRESH_BY_LAYER_ONEBASED.items():
    print(f"  Layer {int(L1)}: tau={tau:.2f}")

rows = []
for layer_idx in layers_to_use:
    layer_onebased = int(layer_idx) + 1
    tau = float(THRESH_BY_LAYER_ONEBASED.get(layer_onebased, 0.0))  # default tau=0 if not specified

    d_cl = _delta_per_run_per_layer(dfm_cl, score_cl, layer_idx, LATE_START_CYCLE, LATE_END_CYCLE)
    d_w  = _delta_per_run_per_layer(dfm_w,  score_w,  layer_idx, LATE_START_CYCLE, LATE_END_CYCLE)

    res = _compare_improvement_gt_tau(d_cl, d_w, tau=tau, prefer_paired=PREFER_PAIRED_IF_POSSIBLE)

    rows.append({
        "Layer": layer_onebased,
        "tau_improvement": res["tau"],
        "test_mode": res["mode"],
        "n_CL": res["n_cl"],
        "n_Walk": res["n_walk"],
        "mean_sep_CL": res["mean_cl"],
        "mean_sep_Walk": res["mean_walk"],
        "mean_improvement": res["mean_diff"],  # mean(Δ_CL - Δ_Walk)
        "p_one_sided_improvement_gt_tau": res["p_one_sided_impr_gt_tau"],
        "t_stat_improvement_gt_tau": res["t_stat_impr_gt_tau"],
        # “inverse”: improvement you can claim with 99% confidence
        "LCB99_mean_improvement": res["lcb99_mean_diff"],
        # optional: 99% two-sided CI
        "CI99_low": res["ci99_low"],
        "CI99_high": res["ci99_high"],
        "df_used": res["df_used"],
    })

df_out = pd.DataFrame(rows).sort_values("Layer").reset_index(drop=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.precision", 6)

print("\n" + "=" * 130)
print("CL vs Walk — Layer-wise improvement thresholds + 99% confidence bounds")
print("Interpretation:")
print("  - p_one_sided_improvement_gt_tau is the one-sided p-value for H1: E[Δ_CL-Δ_Walk] > tau.")
print("  - LCB99_mean_improvement is the 99% one-sided lower bound on E[Δ_CL-Δ_Walk].")
print("=" * 130)
print(df_out)
print("=" * 130)

print("\n[DONE]")


[LOAD] root=/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/WalkSpinJump_relu
[LOAD] Found 10 phase_summary.csv files
[LOAD] Loaded 10 runs successfully
[LOAD] Skipped 0 runs
[LOAD] Behaviors used: ['walk', 'jump', 'spin'] (N=3)
[LOAD] Runs: 10 | Rows(dfm): 4802 | Layers: [0, 1]

[LOAD] root=/Users/adi/Desktop/EmergentRobotSelf/Checkpoints_States_selectedGraphs/Walk_robustness
[LOAD] Found 8 phase_summary.csv files
[LOAD] Loaded 8 runs successfully
[LOAD] Skipped 0 runs
[LOAD] Behaviors used: ['walk'] (N=1)
[LOAD] Runs: 8 | Rows(dfm): 4520 | Layers: [0, 1]

[LATE WINDOW]
  cycles in [15, 50) (i.e., 15..49)
[THRESHOLDS]
  Layer 1: tau=0.10
  Layer 2: tau=0.05

CL vs Walk — Layer-wise improvement thresholds + 99% confidence bounds
Interpretation:
  - p_one_sided_improvement_gt_tau is the one-sided p-value for H1: E[Δ_CL-Δ_Walk] > tau.
  - LCB99_mean_improvement is the 99% one-sided lower bound on E[Δ_CL-Δ_Walk].
   Layer  tau_improvement test_mode  n_CL  n_Walk  m